# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "default"  # 'template' or 'default'
OUTPUT_DIR = "unconstrained_derived_features_review"
RAW_EQS = [
    "((sqrt(P_dyn - -0.887826) * VBs) - square(0.6033792 * (DST + -22.420992))) / (DST + -825.26373)",
    "(VBs * (sqrt(P_dyn + square((DST * -0.008792949) + -1.9299145)) * -0.0009785303)) + square((DST * 0.01791331) + -0.60905385)",
    "square(0.5820395 + (DST * -0.018421166)) - ((sqrt(P_dyn + 1.1339003) * VBs) * 0.001091036)",
    "((VBs * -0.0011102495) * sqrt(P_dyn + 1.0975075)) + (0.00036096844 * square(DST + -29.48951))",
]

start_eq_number = 4

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [3]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

In [4]:
def predict_and_plot_storm(
    model, eqs, start, end, storm_df, storm_id, save_path, eq_start_index=0
):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values
    colors = ["blue", "yellow", "green", "orange", "purple", "cyan", "magenta"]
    res_eqs = []

    metrics_info = []

    for eq_index, eq in enumerate(eqs):
        res_eq = simulate_storm(model[eq], storm_df)
        res_eq = res_eq[start:end]["DST_pred"].values
        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        metrics_info.append(m_eq)
        # string_title += f"Evaluation for Equation {eq_index + 1} ({colors[eq_index]}): ${model[eq].latex_str()}$ \n"

        res_eqs.append(res_eq)

    # 2. Calculate Metrics

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)

    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )

    for eq_index, res_eq in enumerate(res_eqs):

        axs[0].plot(
            storm_df[start:end].index,
            res_eqs[eq_index],
            color=colors[eq_index],
            linestyle="--",
            # label=f"Equation {eq_index + eq_start_index}",
            label=None,
            linewidth=1.5,
        )

    axs[0].tick_params(axis="both", which="major", labelsize=14)
    axs[0].tick_params(axis="both", which="minor", labelsize=10)
    axs[0].legend(fontsize=16)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_title("Storm Reconstruction", fontsize=18)

    axs[0].xaxis.set_major_locator(MultipleLocator(2))

    diffs = []

    for eq_index, res_eq in enumerate(res_eqs):
        diff_eq = res_eq - y_true
        diffs.append(diff_eq)

        # axs[1].plot(storm_df[start:end].index, diff_eq, color=colors[eq_index], label=f"Equation {eq_index + eq_start_index} Error")
        axs[1].plot(
            storm_df[start:end].index, diff_eq, color=colors[eq_index], label=None
        )

    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = f"Error Comparison\n"

    for eq_index, m_eq in enumerate(metrics_info):
        title_metrics += f"Eq {eq_index + eq_start_index} ({colors[eq_index]}): MAE={m_eq['MAE']:.2f}, RMSE={m_eq['RMSE']:.2f}, R²={m_eq['R2']:.3f}, BFE={m_eq['BFE']:.3f}\n"

    # axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_title("Equation error", fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis="both", which="major", labelsize=14)
    axs[1].tick_params(axis="both", which="minor", labelsize=10)

    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        res_eqs,
        [f"Equation {i+ eq_start_index}" for i in range(len(eqs))],
        [colors[i] for i in range(len(eqs))],
        fontsize=16,
        plot_legend=False,
    )

    string_title = f"Storm {storm_id} Reconstruction\n{title_metrics}"
    fig.suptitle(string_title, fontsize=18)
    plt.savefig(save_path)
    plt.close()

In [5]:
def save_prediction_data(model, eqs, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eqs = []
    for eq_index, eq in enumerate(eqs):
        pred_dst_eq = simulate_storm(model[eq], storm_df)
        if model[eq].is_template:
            pred_dst_eq = pred_dst_eq[start:end][
                ["DST_pred", "dDST", "injection_component", "decay_component"]
            ]
        else:
            pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
        pred_dst_eqs.append(pred_dst_eq)
        
    # 3. Baseline Predictions (Burton & OBM)
    
    # 4. Construct Comprehensive DataFrame

    if model[eq].is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values
            results_df[f"Injection_Component_{eq_index+1}"] = pred_dst_eq["injection_component"].values
            results_df[f"Decay_Component_{eq_index+1}"] = pred_dst_eq["decay_component"].values
        
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values


    results_df.to_csv(output_path)
    return results_df

## Test storms

In [6]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
        eq_start_index=start_eq_number
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'a') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(models[RAW_EQ].latex_str())}\n')

100%|██████████| 20/20 [00:17<00:00,  1.12it/s]


In [7]:
storms[0].columns

Index(['Observed_DST', 'Real_dDST_dt', 'Pred_DST_Equation_1',
       'Pred_dDST_dt_Equation_1', 'Pred_DST_Equation_2',
       'Pred_dDST_dt_Equation_2', 'Pred_DST_Equation_3',
       'Pred_dDST_dt_Equation_3', 'Pred_DST_Equation_4',
       'Pred_dDST_dt_Equation_4'],
      dtype='object')

In [8]:
metrics = ["RMSE", "MAE", "R2", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[storm_indices[storm_index], "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]


global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'

for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_BFE,Equation 3_RMSE,Equation 3_MAE,Equation 3_R2,Equation 3_BFE,Equation 4_RMSE,Equation 4_MAE,Equation 4_R2,Equation 4_BFE
54,54,8.06495,6.12139,0.839467,11.308646,8.314912,6.188954,0.829362,11.418362,9.008634,7.113426,0.799701,11.571721,8.779165,6.908173,0.809775,11.484946
55,55,15.909189,12.481491,0.790824,18.405616,16.489959,12.970961,0.775273,18.668759,17.375463,13.731527,0.750489,19.559242,17.007616,13.515262,0.760942,19.159893
56,56,12.654988,9.273064,0.648448,15.835122,12.133223,9.310718,0.676839,14.531033,13.363784,11.014805,0.607965,15.721662,13.103475,10.590417,0.623089,15.541541
57,57,10.005577,7.705164,0.745485,10.435467,10.129723,8.054942,0.739129,11.340401,9.540979,7.609833,0.768572,12.025851,9.483605,7.530154,0.771347,11.574112
58,58,9.828411,7.555463,0.766595,14.806299,9.42175,7.334547,0.78551,14.755529,9.19216,6.387633,0.795836,15.625816,9.198578,6.490953,0.795551,15.365378
59,59,13.734735,11.745709,0.810036,9.730955,13.851601,11.906246,0.80679,10.250493,12.169085,10.187595,0.850877,9.420286,12.340322,10.402339,0.84665,9.42057
60,60,19.322303,13.056422,0.732634,29.309573,17.574061,12.932002,0.778826,24.013474,18.267135,11.955447,0.761037,28.250562,18.338374,12.103383,0.75917,28.23741
61,61,10.938646,7.963481,0.928666,14.228484,12.772821,8.617351,0.902738,17.511873,10.704728,8.287751,0.931685,14.32901,10.645508,8.227253,0.932438,13.88645
62,62,12.965947,10.539221,0.609154,12.92504,13.310303,11.044571,0.588118,12.626724,11.35094,9.076804,0.700456,12.9414,11.64289,9.345714,0.684849,12.910975
63,63,13.845406,10.90738,0.860726,17.904959,15.512114,12.493567,0.825177,20.096805,14.785816,12.000925,0.841164,17.960444,14.659398,11.92777,0.843869,18.078444


In [9]:
print(summary_df.to_latex(index=False, float_format="%.3f").replace("_", " ").replace("Equation ", "Eq ").replace("Storm Index", "Storm"))

\begin{tabular}{lllllllllllllllll}
\toprule
Storm & Eq 1 RMSE & Eq 1 MAE & Eq 1 R2 & Eq 1 BFE & Eq 2 RMSE & Eq 2 MAE & Eq 2 R2 & Eq 2 BFE & Eq 3 RMSE & Eq 3 MAE & Eq 3 R2 & Eq 3 BFE & Eq 4 RMSE & Eq 4 MAE & Eq 4 R2 & Eq 4 BFE \\
\midrule
54 & 8.065 & 6.121 & 0.839 & 11.309 & 8.315 & 6.189 & 0.829 & 11.418 & 9.009 & 7.113 & 0.800 & 11.572 & 8.779 & 6.908 & 0.810 & 11.485 \\
55 & 15.909 & 12.481 & 0.791 & 18.406 & 16.490 & 12.971 & 0.775 & 18.669 & 17.375 & 13.732 & 0.750 & 19.559 & 17.008 & 13.515 & 0.761 & 19.160 \\
56 & 12.655 & 9.273 & 0.648 & 15.835 & 12.133 & 9.311 & 0.677 & 14.531 & 13.364 & 11.015 & 0.608 & 15.722 & 13.103 & 10.590 & 0.623 & 15.542 \\
57 & 10.006 & 7.705 & 0.745 & 10.435 & 10.130 & 8.055 & 0.739 & 11.340 & 9.541 & 7.610 & 0.769 & 12.026 & 9.484 & 7.530 & 0.771 & 11.574 \\
58 & 9.828 & 7.555 & 0.767 & 14.806 & 9.422 & 7.335 & 0.786 & 14.756 & 9.192 & 6.388 & 0.796 & 15.626 & 9.199 & 6.491 & 0.796 & 15.365 \\
59 & 13.735 & 11.746 & 0.810 & 9.731 & 13.852 & 11.906 &

## Train storms

In [ ]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
        eq_start_index=start_eq_number
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)


In [ ]:
metrics = ["RMSE", "MAE", "R2", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[len(summary_df), "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

display(summary_df.mean())

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'



for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

In [ ]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

In [ ]:

storms = range(54, 74)
parent_folder = 'unconstrained-derived-figures'

for storm_number in storms:
    # We use f-strings with double {{ }} to escape the LaTeX braces
    # and single { } for the Python variables.
    latex_code = f"""
\\begin{{figure}}[ht]
    \\centering
    \\includegraphics[width=\\textwidth]{{{parent_folder}/storm_{storm_number}.png}}
    \\caption{{Reconstruction of storm {storm_number} using the Equations generated from the unconstrained symbolic regression with the derived features}}\\label{{fig:unconstrained-storm-{storm_number}}}
\\end{{figure}}
"""
    print(latex_code)